In [8]:
"""
================================================================================
Person A — Dataset Lead
L.O.V.E. Relationship Support Agent | RSM 8430 Group 18
================================================================================

WHO THIS FILE IS FOR
--------------------
This is Person A's only script. Run it once and hand the output to Person B.
Person B does not need to touch this file at all.


WHAT THIS SCRIPT DOES
---------------------
Takes the raw CounselChat dataset (~2,780 rows) from HuggingFace and produces
a clean, filtered CSV that Person B will use to build the ChromaDB vector index.

The raw dataset has one row per therapist answer. The same question can have
10-20 different therapist answers. This script collapses them down to one
best answer per question, then filters to keep only romantic relationship
questions where the relationship is actually the main problem.


HOW TO RUN
----------
Step 1 — Install dependencies (only needed once):
    pip install datasets pandas

Step 2 — Run:
    python person_a_dataset_lead.py

Step 3 — Check the printed summary at the bottom. Look for:
    - Total document count (aim for 200-300)
    - Tier 1 vs Tier 2 split
    - Project topic breakdown

Step 4 — Send the entire data/ folder to Person B.


OUTPUT FILES
------------
data/filtered_counselchat.csv  <-- MAIN FILE. Person B reads this.
data/sample_20_rows.csv        <-- 20 random rows. Open this to spot-check quality.
data/filtering_notes.txt       <-- Plain English explanation of all filter rules.
data/filter_summary.json       <-- Stats (doc count, topic counts, etc.)


WHAT'S IN THE OUTPUT CSV
------------------------
Each row = one unique question + its single best therapist answer.

Columns:
  doc_id          unique ID we assign: cc_0000, cc_0001, ...
  questionID      original CounselChat question ID
  question_title  short title of the question
  question_text   full question body
  answer_text     best therapist answer (highest upvotes, then longest)
  original_topic  CounselChat's own category label (e.g. "relationships")
  project_topic   our label: breakup / trust / communication / conflict /
                  marriage / intimacy / boundaries / commitment /
                  emotional_distance / relationship_general
  tier            1 = high confidence, 2 = extended (see FILTERING LOGIC below)
  upvotes         upvotes on the kept answer
  ans_len         character length of the answer
  document_text   THE STRING PERSON B EMBEDS INTO CHROMADB.
                  Format: "Question Title: ...\n\nQuestion: ...\n\nTherapist Answer: ..."


FILTERING LOGIC
---------------
We use a two-tier system to balance quality vs. coverage.

TIER 1 — Strict filter (~113 rows, high confidence)
  Keep a row only if BOTH are true:
    (a) Question explicitly mentions a romantic partner
        e.g. "my boyfriend", "my husband", "my ex", "our relationship"
        We use possessive phrases, not bare words like "boyfriend", because
        bare words often appear in therapist answers for unrelated topics.
    (b) The relationship is clearly the core subject
        e.g. "we fight", "he cheated", "should I break up", "feel unloved"
        This filters out rows where a partner is mentioned but the real
        problem is anxiety, depression, career stress, identity crisis, etc.
  Then we apply BAD_PATTERNS to remove any remaining obvious noise
  (e.g. "how can I manage my anxiety", "I hate myself").

TIER 2 — Loose filter (~112 rows, extended coverage)
  Same partner condition as Tier 1, but the relationship signal is softer.
  We pick up valid questions that don't use strong conflict words, like:
    "My boyfriend doesn't compromise with me"
    "How do I make my relationship with my girlfriend better?"
  BAD_PATTERNS is also applied here to keep the noise low.

Both tiers are merged into one CSV. The tier column tells Person B which
is which. Suggested usage for Person B:
  - Default: use ALL rows for RAG (better coverage)
  - If retrieval feels noisy: filter to tier=1 only (higher precision)


HOW project_topic IS ASSIGNED
------------------------------
We count keyword hits for each topic across the question + answer text,
then assign the topic with the highest count.

This avoids the "first match wins" problem. For example, a row about a
breakup where the word "trust" appears once in the answer should be labelled
"breakup", not "trust". Scoring by count fixes this.
"""

from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from datasets import load_dataset


# ================================================================================
# Config
# ================================================================================

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_OUTPUT = OUTPUT_DIR / "filtered_counselchat.csv"
SAMPLE_OUTPUT   = OUTPUT_DIR / "sample_20_rows.csv"
NOTES_OUTPUT    = OUTPUT_DIR / "filtering_notes.txt"
SUMMARY_OUTPUT  = OUTPUT_DIR / "filter_summary.json"

# Answers shorter than this are usually one-liners with no real content.
# 80 chars ≈ one short sentence.
MIN_ANSWER_CHARS = 80


# ================================================================================
# Hard exclusions
#
# Applied before any other filter. These are dropped immediately because the
# agent should never attempt to handle them — they need professional crisis support.
# Note: we only exclude true crisis content. Things like "bipolar" or "hospitalized"
# are NOT excluded because they often appear in valid relationship questions
# (e.g. "my husband has bipolar and it's affecting our relationship").
# ================================================================================

HARD_EXCLUDE = [
    "suicide", "suicidal", "kill myself", "end my life", "take my life",
    "self-harm", "self harm", "cutting myself", "hurt myself",
    "rape", "sexual assault", "molested",
    "cancer", "tumor", "chemotherapy", "terminal illness",
]


# ================================================================================
# BAD_PATTERNS
#
# Applied after the tier filters as a final cleanup pass.
# These are patterns where a romantic partner is mentioned but the core question
# is clearly about personal psychology, not the relationship itself.
#
# Examples of what this removes:
#   "How can I manage my anxiety?" (partner mentioned in the body, but not the point)
#   "I find myself lying about small things" (honesty/self-awareness, not relationship)
#   "My father is an alcoholic and I recently got married" (core = addiction, not marriage)
# ================================================================================

BAD_PATTERNS = [
    # Pure mental health management — relationship is just context
    "how can i manage my anxiety",
    "how do i manage my anxiety",
    "how do i be happy",
    "how can i be happy",
    "how do i feel better",
    "how can i feel better",
    "i hate myself",
    "i have no motivation",
    "all i can do is cry",
    "i can't stop crying",
    # Identity / self-worth questions with no relationship interaction
    "finding my identity",
    "find my true identity",
    "lost my identity",
    "learn to be content",
    "learn to love myself",
    "feel like my life is out of control",
    "life is out of control",
    # Family dynamics with no romantic partner as the focus
    "why did my sister",
    "why did my brother",
    "why did my mother",
    "why did my father",
    # Post-breakup personal healing (not relationship advice)
    "how can i learn to be content",
    "how do i move on with my life",
    "how do i stop thinking about",
    # Specific noisy patterns we caught in manual review
    "lying about", "why do i lie", "i find myself lying",
    "alcoholic", "drinking problem", "willpower to stop",
    "dog obsession", "obsession with",
    "friends who don't", "friendship",
    "get on base", "military base", "id card",
    "no need to lie",
]


# ================================================================================
# Condition 1 — Partner keywords (used by both tiers)
#
# At least one of these must appear in question title + question text combined.
# We use possessive phrases ("my boyfriend") instead of bare words ("boyfriend")
# because bare words appear constantly in therapist answers for unrelated topics.
# ================================================================================

PARTNER_KEYWORDS = [
    "my boyfriend", "my girlfriend", "my husband", "my wife",
    "my fiance", "my fiancé", "my fiancee", "my fiancée",
    "my partner", "my spouse", "my ex",
    "my ex-boyfriend", "my ex-girlfriend",
    "my ex-husband", "my ex-wife",
    "our relationship", "my marriage",
    "we are dating", "we have been dating",
    "we are in a relationship", "been together",
    "we got married", "we got engaged",
    "my significant other",
    "boyfriend and i", "girlfriend and i",
    "husband and i", "wife and i",
]


# ================================================================================
# Condition 2 STRICT — used for Tier 1
#
# The relationship must be the clear core subject of the question.
# These are unambiguous signals that the person is asking about the relationship
# itself, not just mentioning a partner while asking about something else.
# ================================================================================

STRICT_CORE_KEYWORDS = [
    # Active conflict
    "we fight", "we argue", "we argued", "we keep fighting",
    "always arguing", "always fighting",
    "had an argument", "had a fight", "big fight",
    # Trust / cheating
    "he cheated", "she cheated", "they cheated",
    "he is cheating", "she is cheating",
    "caught him", "caught her", "found out he", "found out she",
    "trust issues", "can't trust", "don't trust",
    "he lied", "she lied", "he lies", "she lies",
    "affair", "cheating", "infidelity", "betrayal",
    # Breakup / separation
    "broke up", "break up", "breaking up",
    "broke up with me", "broke up with him", "broke up with her",
    "wants to break up", "thinking of breaking up",
    "ended the relationship", "ended our relationship",
    "separated", "separation",
    "divorce", "divorcing", "want a divorce",
    "left me", "he left", "she left", "she left me", "he left me",
    # Communication problems
    "won't talk", "doesn't talk", "refuses to talk",
    "won't communicate", "shuts down", "stonewalls",
    "silent treatment", "hard to communicate",
    # Advice-seeking about the relationship
    "how do i talk to my", "how should i talk to",
    "how do i bring up", "how do i approach my",
    "how do i tell my", "how can i tell my",
    "should i break up", "should we break up",
    "should i leave", "should i stay",
    "should i marry", "should i divorce",
    "how to save my relationship", "save our relationship",
    "how to fix our relationship", "make it work",
    "give him another chance", "give her another chance",
    "take him back", "take her back", "get back together",
    # Emotional signals about the relationship
    "feel unloved", "feeling unloved",
    "feel neglected", "feeling neglected",
    "feel unwanted", "feel unappreciated",
    "feel trapped in", "unhappy in my relationship",
    "unhappy in our relationship",
    "falling out of love", "fell out of love",
    "don't love him anymore", "don't love her anymore",
    "not in love with", "losing feelings", "lost feelings",
    "drifting apart", "growing apart",
    "feel disconnected", "feel alone in my relationship",
    "feel like roommates",
    # Behavioral issues in the relationship
    "jealous", "jealousy", "controlling", "possessive", "manipulative",
    "emotionally abusive", "verbal abuse",
    "he ignores me", "she ignores me", "ignoring me",
    "pulling away",
    # Intimacy
    "no intimacy", "lack of intimacy", "no affection", "lack of affection",
    "not attracted", "lost attraction", "sex life", "sexual problems",
    # Commitment
    "won't commit", "doesn't want to commit",
    "different goals", "not on the same page",
    "get over my ex", "get over him", "get over her",
    "still love my ex", "miss my ex",
    "after the breakup", "since we broke up",
]


# ================================================================================
# Condition 2 LOOSE — used for Tier 2
#
# Softer signals. Picks up valid questions that don't use explicit conflict words
# but are still clearly about the relationship, for example:
#   "My boyfriend doesn't compromise with me" → hits "he doesn't"
#   "How do I make my relationship with my girlfriend better?" → hits "how do i"
# ================================================================================

LOOSE_CORE_KEYWORDS = [
    "relationship", "together", "dating", "in love",
    "feelings for", "how do i", "should i", "what should i do",
    "how can i", "i need help with", "i don't know what to do",
    "confused about", "not sure if", "worried about",
    "he doesn't", "she doesn't", "he never", "she never",
    "he always", "she always", "he keeps", "she keeps",
    "he wants", "she wants", "he said", "she said",
    "we don't", "we never", "we always", "we have",
    "how to deal with", "how to handle",
    "is it normal", "is this healthy", "is this okay",
    "red flag", "toxic", "unhealthy",
    "long distance", "long-distance",
    "open relationship", "polyamory",
]


# ================================================================================
# Project topic scoring
#
# We count keyword hits for each topic across question + answer text combined,
# then assign the topic with the highest count.
#
# Why scoring instead of first-match:
#   "My ex-boyfriend is with someone new and I can't trust anyone"
#   → trust keywords: 1 hit ("trust")
#   → breakup keywords: 3 hits ("ex-boyfriend", "my ex", "someone new")
#   → correctly labelled "breakup", not "trust"
# ================================================================================

PROJECT_TOPIC_MAP = {
    "breakup": [
        "broke up", "break up", "breaking up", "my ex", "ex-boyfriend",
        "ex-girlfriend", "ex-husband", "ex-wife", "get over him", "get over her",
        "get over my ex", "miss my ex", "still love my ex", "take him back",
        "take her back", "get back together", "after the breakup",
        "since we broke up", "move on", "he left", "she left",
    ],
    "trust": [
        "cheating", "cheated", "affair", "infidelity", "betrayal",
        "trust issues", "can't trust", "don't trust", "he lied", "she lied",
        "he lies", "she lies", "jealous", "jealousy", "caught him", "caught her",
    ],
    "communication": [
        "communication", "communicate", "won't talk", "doesn't talk",
        "silent treatment", "shuts down", "stonewalls",
        "how do i talk to", "how do i bring up", "hard to communicate",
        "how do i tell my",
    ],
    "conflict": [
        "we fight", "we argue", "we argued", "argument",
        "always fighting", "always arguing", "had a fight", "had an argument",
    ],
    "marriage": [
        "marriage", "married", "husband", "wife", "spouse",
        "wedding", "divorce", "divorcing", "want a divorce", "should i marry",
    ],
    "intimacy": [
        "intimacy", "intimate", "affection", "closeness",
        "sex life", "sexual problems", "not attracted", "lost attraction",
        "no affection", "lack of affection",
    ],
    "boundaries": [
        "controlling", "possessive", "manipulative", "boundary", "boundaries",
        "emotionally abusive", "verbal abuse", "toxic",
    ],
    "commitment": [
        "won't commit", "doesn't want to commit", "different goals",
        "not on the same page", "doesn't want kids", "doesn't want marriage",
        "scared of commitment", "fear of commitment", "long distance",
    ],
    "emotional_distance": [
        "drifting apart", "growing apart", "feel disconnected", "distant",
        "feel alone in my relationship", "feel like roommates",
        "falling out of love", "lost feelings", "losing feelings",
        "don't love him anymore", "don't love her anymore",
        "feel unloved", "feel neglected", "feel unwanted",
        "unhappy in my relationship",
    ],
}


# ================================================================================
# Helpers
# ================================================================================

def norm(text) -> str:
    """Lowercase and collapse whitespace. Returns empty string for non-strings."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text.lower()).strip()


def has_any(text: str, keywords: list[str]) -> bool:
    """Return True if any keyword appears in text."""
    return any(kw in text for kw in keywords)


def assign_project_topic(q_norm: str, a_norm: str) -> str:
    """
    Score each topic by counting how many of its keywords appear in
    question + answer combined. Return the highest-scoring topic.
    Falls back to "relationship_general" if nothing matches.
    """
    combined = q_norm + " " + a_norm
    scores = {
        topic: sum(1 for kw in keywords if kw in combined)
        for topic, keywords in PROJECT_TOPIC_MAP.items()
    }
    best = max(scores, key=lambda t: scores[t])
    return best if scores[best] > 0 else "relationship_general"


def build_document(title: str, question: str, answer: str) -> str:
    """
    Build the string that Person B will embed into ChromaDB.
    Putting the title first helps semantic search match on the core question.
    """
    title = title.strip() if isinstance(title, str) else ""
    parts = []
    if title:
        parts.append(f"Question Title: {title}")
    parts.append(f"Question: {question.strip()}")
    parts.append(f"Therapist Answer: {answer.strip()}")
    return "\n\n".join(parts)


# ================================================================================
# Main pipeline
# ================================================================================

def main():

    # ── Step 1: Load the dataset ──────────────────────────────────────────────
    # Try the datasets library first (handles caching and auth automatically).
    # Falls back to direct CSV if that fails.
    print("Loading CounselChat from HuggingFace...")
    try:
        raw = load_dataset("nbertagnolli/counsel-chat", split="train")
        df  = raw.to_pandas()
        print("  Loaded via datasets library.")
    except Exception as e:
        print(f"  datasets failed ({e}), trying direct CSV...")
        df = pd.read_csv(
            "hf://datasets/nbertagnolli/counsel-chat/20220401_counsel_chat.csv"
        )

    print(f"  Raw rows:     {len(df)}")
    print(f"  Unique Qs:    {df['questionID'].nunique()}")
    print()

    # ── Step 2: Normalize columns ─────────────────────────────────────────────
    # Fill missing columns with safe defaults in case the schema changes.
    for col, default in [("questionTitle", ""), ("topic", ""), ("upvotes", 0)]:
        if col not in df.columns:
            df[col] = default

    df = df.dropna(subset=["questionID", "questionText", "answerText"]).copy()

    # Lowercase normalized versions used for all keyword matching below.
    # We never modify the original columns.
    df["title_n"]   = df["questionTitle"].apply(norm)
    df["q_n"]       = df["questionText"].apply(norm)
    df["a_n"]       = df["answerText"].apply(norm)
    df["topic_n"]   = df["topic"].apply(norm)
    df["ans_len"]   = df["answerText"].str.len()
    df["upvotes_n"] = pd.to_numeric(df["upvotes"], errors="coerce").fillna(0)

    # Drop very short answers — they're usually one-liners with no useful content.
    before = len(df)
    df = df[df["ans_len"] >= MIN_ANSWER_CHARS].copy()
    print(f"Dropped {before - len(df)} short answers (<{MIN_ANSWER_CHARS} chars). Remaining: {len(df)}")

    # Drop hard-excluded crisis content.
    before = len(df)
    df = df[~df["q_n"].apply(lambda x: has_any(x, HARD_EXCLUDE))].copy()
    print(f"Dropped {before - len(df)} crisis rows. Remaining: {len(df)}")

    # ── Step 3: Dedup — one best answer per question ──────────────────────────
    # Sort so the highest-upvoted, longest answer comes first within each
    # questionID group. drop_duplicates then keeps only that first row.
    # We do this BEFORE the tier filters so each question is only evaluated once.
    df = (
        df.sort_values(
            by=["questionID", "upvotes_n", "ans_len"],
            ascending=[True, False, False]
        )
        .drop_duplicates(subset=["questionID"], keep="first")
        .copy()
    )
    print(f"After dedup: {len(df)} unique questions")
    print()

    # Combined field used for all keyword matching below.
    df["q_combined"] = df["title_n"] + " " + df["q_n"]

    # ── Step 4: Tier 1 — strict filter ───────────────────────────────────────
    # Both conditions must be true.
    cond1        = df["q_combined"].apply(lambda x: has_any(x, PARTNER_KEYWORDS))
    cond2_strict = df["q_combined"].apply(lambda x: has_any(x, STRICT_CORE_KEYWORDS))
    df_tier1_raw = df[cond1 & cond2_strict].copy()
    print(f"Tier 1 raw (before BAD_PATTERNS clean): {len(df_tier1_raw)}")

    # ── Step 5: BAD_PATTERNS clean on Tier 1 ─────────────────────────────────
    # Remove rows that passed the strict filter but are still clearly about
    # personal psychology rather than the relationship itself.
    bad_mask = df_tier1_raw["q_combined"].apply(lambda x: has_any(x, BAD_PATTERNS))
    df_tier1 = df_tier1_raw[~bad_mask].copy()
    df_tier1["tier"] = 1
    print(f"Tier 1 after BAD_PATTERNS clean:        {len(df_tier1)}  (removed {bad_mask.sum()} rows)")
    print()

    # ── Step 6: Tier 2 — loose filter on remaining rows ──────────────────────
    # Only rows NOT already in Tier 1 are considered here.
    # Same partner condition, but softer relationship signal.
    tier1_ids    = set(df_tier1["questionID"])
    df_remaining = df[~df["questionID"].isin(tier1_ids)].copy()

    cond1_r      = df_remaining["q_combined"].apply(lambda x: has_any(x, PARTNER_KEYWORDS))
    cond2_loose  = df_remaining["q_combined"].apply(lambda x: has_any(x, LOOSE_CORE_KEYWORDS))
    df_tier2_raw = df_remaining[cond1_r & cond2_loose].copy()

    # Apply BAD_PATTERNS to Tier 2 as well.
    bad_mask2 = df_tier2_raw["q_combined"].apply(lambda x: has_any(x, BAD_PATTERNS))
    df_tier2  = df_tier2_raw[~bad_mask2].copy()
    df_tier2["tier"] = 2
    print(f"Tier 2 (loose filter, cleaned):         {len(df_tier2)}")
    print()

    # ── Step 7: Merge ─────────────────────────────────────────────────────────
    df_all = pd.concat([df_tier1, df_tier2], ignore_index=True)
    print(f"Total combined: {len(df_all)}")
    print()

    # ── Step 8: Assign project topic ──────────────────────────────────────────
    df_all["project_topic"] = df_all.apply(
        lambda r: assign_project_topic(r["q_n"], r["a_n"]), axis=1
    )

    # ── Step 9: Build document_text ───────────────────────────────────────────
    # This is the string Person B passes to the embedding model.
    df_all["document_text"] = df_all.apply(
        lambda r: build_document(
            r.get("questionTitle", ""),
            r["questionText"],
            r["answerText"],
        ),
        axis=1,
    )

    # ── Step 10: Assemble final dataframe ─────────────────────────────────────
    df_all = df_all.reset_index(drop=True)
    df_all["doc_id"] = df_all.index.map(lambda i: f"cc_{i:04d}")

    df_final = df_all[[
        "doc_id",
        "questionID",
        "questionTitle",
        "questionText",
        "answerText",
        "topic",
        "project_topic",
        "tier",
        "upvotes_n",
        "ans_len",
        "document_text",
    ]].rename(columns={
        "questionTitle": "question_title",
        "questionText":  "question_text",
        "answerText":    "answer_text",
        "topic":         "original_topic",
        "upvotes_n":     "upvotes",
    })

    # ── Step 11: Print summary ────────────────────────────────────────────────
    print("=" * 55)
    print("HANDOFF SUMMARY FOR PERSON B")
    print("=" * 55)
    print(f"  Tier 1 (high confidence): {(df_final['tier']==1).sum()}")
    print(f"  Tier 2 (extended):        {(df_final['tier']==2).sum()}")
    print(f"  Total:                    {len(df_final)}")

    if len(df_final) < 50:
        print(f"  ❌ Below rubric minimum (50). Something is wrong with the filters.")
    elif len(df_final) < 300:
        print(f"  ⚠️  Below proposal target (300). Tier 1 alone is fine for demo.")
    else:
        print(f"  ✅ At or above proposal target (300).")

    print(f"\n  Project topic breakdown:")
    for topic, count in df_final["project_topic"].value_counts().items():
        print(f"    {topic:<25} {count}")

    print(f"\n  Original CounselChat topic breakdown:")
    for topic, count in df_final["original_topic"].value_counts().items():
        print(f"    {topic:<25} {count}")

    print(f"\n  Avg answer length: {df_final['ans_len'].mean():.0f} chars")
    print(f"  Avg upvotes:       {df_final['upvotes'].mean():.2f}")

    # ── Step 12: Save ─────────────────────────────────────────────────────────
    df_final.to_csv(FILTERED_OUTPUT, index=False)
    df_final.sample(min(20, len(df_final)), random_state=42).to_csv(
        SAMPLE_OUTPUT, index=False
    )

    summary = {
        "tier1_count":           int((df_final["tier"] == 1).sum()),
        "tier2_count":           int((df_final["tier"] == 2).sum()),
        "total_documents":       len(df_final),
        "project_topic_counts":  df_final["project_topic"].value_counts().to_dict(),
        "original_topic_counts": df_final["original_topic"].value_counts().to_dict(),
        "min_answer_chars":      MIN_ANSWER_CHARS,
    }
    SUMMARY_OUTPUT.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    notes = f"""Person A — Filtering Notes
Dataset: nbertagnolli/counsel-chat (CounselChat)

PIPELINE STEPS
--------------
1. Load raw data (~2,780 rows, ~940 unique questions)
2. Drop answers shorter than {MIN_ANSWER_CHARS} chars (one-liners, no real content)
3. Drop hard crisis content from question text (suicide, rape, cancer)
4. Dedup: sort by upvotes desc + answer length desc, keep first per questionID
5. Tier 1 — strict two-condition filter:
     Cond 1: question mentions a romantic partner (possessive phrases)
     Cond 2: relationship is clearly the core subject (conflict/emotional signals)
   Then BAD_PATTERNS clean: remove obvious personal-psychology noise
6. Tier 2 — loose filter on rows not already in Tier 1:
     Cond 1: same partner keywords
     Cond 2: any softer relationship signal (he said, she wants, how do i, etc.)
   BAD_PATTERNS clean applied here too
7. Merge tiers, assign project_topic by keyword scoring, export

TIER GUIDE FOR PERSON B
------------------------
tier = 1  high confidence — relationship is clearly the core topic
tier = 2  extended — relationship is present but signal is softer

Default: use all rows for RAG (better coverage).
If retrieval feels noisy: filter to tier=1 only (higher precision).
  df = df[df['tier'] == 1]

PROJECT TOPIC ASSIGNMENT
------------------------
We count keyword hits per topic across question + answer combined.
The topic with the highest count wins. Falls back to "relationship_general".
Topics: breakup, trust, communication, conflict, marriage,
        intimacy, boundaries, commitment, emotional_distance

HARD EXCLUSIONS
---------------
Dropped before all filters. Question text must NOT contain:
  suicide / suicidal / kill myself / end my life / take my life
  self-harm / self harm / cutting myself / hurt myself
  rape / sexual assault / molested
  cancer / tumor / chemotherapy / terminal illness

NOT excluded: bipolar, hospitalized, psychiatric — these often appear in
valid relationship questions (e.g. "my husband has bipolar disorder and
it's affecting our marriage").

FINAL COUNT: {len(df_final)} ({(df_final['tier']==1).sum()} tier1 + {(df_final['tier']==2).sum()} tier2)
OUTPUT: {FILTERED_OUTPUT}
"""
    NOTES_OUTPUT.write_text(notes, encoding="utf-8")

    print()
    print(f"  Saved → {FILTERED_OUTPUT}")
    print(f"  Saved → {SAMPLE_OUTPUT}")
    print(f"  Saved → {SUMMARY_OUTPUT}")
    print(f"  Saved → {NOTES_OUTPUT}")
    print()
    print("Done. Send the data/ folder to Person B.")
    print()
    print("Reminder for Person B:")
    print("  - The column to embed is: document_text")
    print("  - tier=1 is high confidence, tier=2 is extended")
    print("  - Start with all rows. If RAG feels noisy, filter to tier=1 only.")


if __name__ == "__main__":
    main()

Loading CounselChat from HuggingFace...


Repo card metadata block was not found. Setting CardData to empty.


  Loaded via datasets library.
  Raw rows:     2775
  Unique Qs:    940

Dropped 11 short answers (<80 chars). Remaining: 2601
Dropped 202 crisis rows. Remaining: 2399
After dedup: 825 unique questions

Tier 1 raw (before BAD_PATTERNS clean): 124
Tier 1 after BAD_PATTERNS clean:        113  (removed 11 rows)

Tier 2 (loose filter, cleaned):         112

Total combined: 225

HANDOFF SUMMARY FOR PERSON B
  Tier 1 (high confidence): 113
  Tier 2 (extended):        112
  Total:                    225
  ⚠️  Below proposal target (300). Tier 1 alone is fine for demo.

  Project topic breakdown:
    marriage                  74
    breakup                   40
    trust                     35
    relationship_general      35
    communication             17
    intimacy                  10
    boundaries                8
    conflict                  5
    emotional_distance        1

  Original CounselChat topic breakdown:
    intimacy                  65
    relationships             56
   